# Etapa 1 — Data Leakage Demo y Comparación de Modelos

**Proyecto Integrador de Aprendizaje Automático — Predicción de Falla Cardíaca**

Basado en la sección **10.12** de las notas *"Pipelines — Machine Learning"*.

Este notebook:

1. Carga el dataset real [`fedesoriano/heart-failure-prediction`](https://www.kaggle.com/datasets/fedesoriano/heart-failure-prediction) (descargado manualmente en `data/heart.csv`).
2. Ilustra el efecto del **data leakage** comparando un flujo con fuga de información vs. uno correcto usando `Pipeline`.
3. Implementa los 5 modelos pedidos (`SVC`, `LogisticRegression`, `RandomForestClassifier`, `KNeighborsClassifier`, `GradientBoostingClassifier`), cada uno dentro de un `Pipeline` + `GridSearchCV`.
4. Compara AUC y Accuracy de todos los modelos y presenta un ranking final.


## 1. Instalación e importación de librerías

In [1]:
# pip install kagglehub[pandas-datasets] scikit-learn pandas numpy matplotlib seaborn

import numpy as np
import pandas as pd

from sklearn.compose import ColumnTransformer
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import MinMaxScaler, OneHotEncoder
from sklearn.svm import SVC
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import roc_auc_score, accuracy_score

RANDOM_STATE = 42


## 2. Carga de datos

El dataset ya fue descargado manualmente desde Kaggle y se encuentra en `data/heart.csv`
(dentro de esta carpeta del proyecto). Se carga directamente con pandas; como alternativa se deja
comentado el código con `kagglehub` que indican las notas del curso, por si se prefiere la descarga
automática (requiere credenciales de Kaggle en `~/.kaggle/kaggle.json` o
`KAGGLE_USERNAME`/`KAGGLE_KEY`).


In [2]:
import pandas as pd

df = pd.read_csv("../data/heart.csv")

# --- Alternativa con kagglehub (descarga automática, requiere credenciales de Kaggle) ---
# import kagglehub
# from kagglehub import KaggleDatasetAdapter
# file_path = "heart.csv"
# df = kagglehub.load_dataset(
#     KaggleDatasetAdapter.PANDAS,
#     "fedesoriano/heart-failure-prediction",
#     file_path,
# )

print(f"Dimensiones: {df.shape[0]} filas x {df.shape[1]} columnas")
df.head()


Dimensiones: 918 filas x 12 columnas


,Age,Sex,ChestPainType,RestingBP,Cholesterol,FastingBS,RestingECG,MaxHR,ExerciseAngina,Oldpeak,ST_Slope,HeartDisease
0,40,M,ATA,140,289,0,Normal,172,N,0.0,Up,0
1,49,F,NAP,160,180,0,Normal,156,N,1.0,Flat,1
2,37,M,ATA,130,283,0,ST,98,N,0.0,Up,0
3,48,F,ASY,138,214,0,Normal,108,Y,1.5,Flat,1
4,54,M,NAP,150,195,0,Normal,122,N,0.0,Up,0


## 3. Columna objetivo y esquema de features

> **Nota respecto al PDF:** el ejemplo de las notas usa una columna genérica `target`, pero el dataset
> real de Kaggle usa la columna **`HeartDisease`** (`1` = riesgo de falla cardíaca, `0` = sin riesgo),
> tal como se describe en el contexto del problema (sección 10.12.1). Usamos `HeartDisease` en todo
> el notebook.

Además, a diferencia del ejemplo sintético del PDF (100% numérico), este dataset real mezcla variables
**numéricas** (`Age`, `RestingBP`, `Cholesterol`, `FastingBS`, `MaxHR`, `Oldpeak`) y **categóricas**
(`Sex`, `ChestPainType`, `RestingECG`, `ExerciseAngina`, `ST_Slope`). Por eso el preprocesamiento no puede
ser un simple `MinMaxScaler()` sobre todo `X`: se usa un `ColumnTransformer` que aplica `OneHotEncoder`
a las categóricas y `MinMaxScaler` a las numéricas, **integrado como primer paso del `Pipeline`** para
evitar fuga de datos también en el encoding.


In [3]:
TARGET_COL = "HeartDisease"

X = df.drop(columns=[TARGET_COL])
y = df[TARGET_COL]

CATEGORICAL_COLS = [c for c in ["Sex", "ChestPainType", "RestingECG", "ExerciseAngina", "ST_Slope"] if c in X.columns]
NUMERIC_COLS = [c for c in X.columns if c not in CATEGORICAL_COLS]

print("Numéricas:", NUMERIC_COLS)
print("Categóricas:", CATEGORICAL_COLS)

preprocessor = ColumnTransformer(
    transformers=[
        ("num", MinMaxScaler(), NUMERIC_COLS),
        ("cat", OneHotEncoder(handle_unknown="ignore"), CATEGORICAL_COLS),
    ]
)


Numéricas: ['Age', 'RestingBP', 'Cholesterol', 'FastingBS', 'MaxHR', 'Oldpeak']
Categóricas: ['Sex', 'ChestPainType', 'RestingECG', 'ExerciseAngina', 'ST_Slope']


## 4. Ilustración de Data Leakage

Reproducimos el ejercicio de la sección 10.12.6 de las notas: se introduce una **variable artificial
con fuga** (`leaky_feature`), altamente correlacionada con el target, y se compara:

1. **Primer intento (con fuga):** se escala/ajusta el preprocesamiento con **todo el dataset** antes de
   dividir en train/test.
2. **Segundo intento (sin fuga, con `Pipeline`):** el `ColumnTransformer` se ajusta **solo con el
   conjunto de entrenamiento**, dentro de un `Pipeline` + `GridSearchCV`.


In [4]:
np.random.seed(RANDOM_STATE)

# Variable artificial con fuga: altamente correlacionada con el target real
X_leak = X.copy()
X_leak["leaky_feature"] = y.values + np.random.normal(0, 0.05, size=len(y))

preprocessor_leak = ColumnTransformer(
    transformers=[
        ("num", MinMaxScaler(), NUMERIC_COLS + ["leaky_feature"]),
        ("cat", OneHotEncoder(handle_unknown="ignore"), CATEGORICAL_COLS),
    ]
)


In [5]:
# --- Primer intento: fuga de datos (preprocesamiento ANTES del split) ---
X_leak_transformed = preprocessor_leak.fit_transform(X_leak, y)

X_train_l, X_test_l, y_train_l, y_test_l = train_test_split(
    X_leak_transformed, y, test_size=0.3, random_state=RANDOM_STATE, stratify=y
)

param_grid_svc = {"C": [0.01, 0.1, 1, 10, 100], "gamma": [0.01, 0.1, 1, 10, 100]}

grid_leak = GridSearchCV(SVC(probability=True), param_grid=param_grid_svc, cv=5, scoring="roc_auc")
grid_leak.fit(X_train_l, y_train_l)

y_scores_leak = grid_leak.predict_proba(X_test_l)[:, 1]
auc_leak = roc_auc_score(y_test_l, y_scores_leak)

print(f"AUC con data leakage (sin pipeline): {auc_leak:.3f}")
print("Mejores parámetros (con leakage):", grid_leak.best_params_)


/home/yisusparker/.local/lib/python3.14/site-packages/sklearn/svm/_base.py:239: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(
/home/yisusparker/.local/lib/python3.14/site-packages/sklearn/svm/_base.py:239: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(
/home/yisusparker/.local/lib/python3.14/site-packages/sklearn/svm/_base.py:239: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(
/home/yisusparker/.local/lib/python3.14/site-packages/sklearn/svm/_base.py:239: FutureWarning: The `probability` parameter was deprecated in 1

/home/yisusparker/.local/lib/python3.14/site-packages/sklearn/svm/_base.py:239: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(
/home/yisusparker/.local/lib/python3.14/site-packages/sklearn/svm/_base.py:239: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(
/home/yisusparker/.local/lib/python3.14/site-packages/sklearn/svm/_base.py:239: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(
/home/yisusparker/.local/lib/python3.14/site-packages/sklearn/svm/_base.py:239: FutureWarning: The `probability` parameter was deprecated in 1

/home/yisusparker/.local/lib/python3.14/site-packages/sklearn/svm/_base.py:239: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(
/home/yisusparker/.local/lib/python3.14/site-packages/sklearn/svm/_base.py:239: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(
/home/yisusparker/.local/lib/python3.14/site-packages/sklearn/svm/_base.py:239: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(
/home/yisusparker/.local/lib/python3.14/site-packages/sklearn/svm/_base.py:239: FutureWarning: The `probability` parameter was deprecated in 1

/home/yisusparker/.local/lib/python3.14/site-packages/sklearn/svm/_base.py:239: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(
/home/yisusparker/.local/lib/python3.14/site-packages/sklearn/svm/_base.py:239: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(
/home/yisusparker/.local/lib/python3.14/site-packages/sklearn/svm/_base.py:239: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(
/home/yisusparker/.local/lib/python3.14/site-packages/sklearn/svm/_base.py:239: FutureWarning: The `probability` parameter was deprecated in 1

/home/yisusparker/.local/lib/python3.14/site-packages/sklearn/svm/_base.py:239: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(
/home/yisusparker/.local/lib/python3.14/site-packages/sklearn/svm/_base.py:239: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(
/home/yisusparker/.local/lib/python3.14/site-packages/sklearn/svm/_base.py:239: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(
/home/yisusparker/.local/lib/python3.14/site-packages/sklearn/svm/_base.py:239: FutureWarning: The `probability` parameter was deprecated in 1

...

/home/yisusparker/.local/lib/python3.14/site-packages/sklearn/svm/_base.py:239: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(
/home/yisusparker/.local/lib/python3.14/site-packages/sklearn/svm/_base.py:239: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(
/home/yisusparker/.local/lib/python3.14/site-packages/sklearn/svm/_base.py:239: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(
/home/yisusparker/.local/lib/python3.14/site-packages/sklearn/svm/_base.py:239: FutureWarning: The `probability` parameter was deprecated in 1

/home/yisusparker/.local/lib/python3.14/site-packages/sklearn/svm/_base.py:239: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(
/home/yisusparker/.local/lib/python3.14/site-packages/sklearn/svm/_base.py:239: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(
/home/yisusparker/.local/lib/python3.14/site-packages/sklearn/svm/_base.py:239: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(
/home/yisusparker/.local/lib/python3.14/site-packages/sklearn/svm/_base.py:239: FutureWarning: The `probability` parameter was deprecated in 1

/home/yisusparker/.local/lib/python3.14/site-packages/sklearn/svm/_base.py:239: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(
/home/yisusparker/.local/lib/python3.14/site-packages/sklearn/svm/_base.py:239: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(
/home/yisusparker/.local/lib/python3.14/site-packages/sklearn/svm/_base.py:239: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(
/home/yisusparker/.local/lib/python3.14/site-packages/sklearn/svm/_base.py:239: FutureWarning: The `probability` parameter was deprecated in 1

AUC con data leakage (sin pipeline): 1.000
Mejores parámetros (con leakage): {'C': 1, 'gamma': 0.1}


/home/yisusparker/.local/lib/python3.14/site-packages/sklearn/svm/_base.py:239: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(
/home/yisusparker/.local/lib/python3.14/site-packages/sklearn/svm/_base.py:239: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(


In [6]:
# --- Segundo intento: flujo correcto con Pipeline (sin fuga) ---
X_train, X_test, y_train, y_test = train_test_split(
    X_leak, y, test_size=0.3, random_state=RANDOM_STATE, stratify=y
)

pipe_leak_free = Pipeline([
    ("preprocessing", preprocessor_leak),
    ("svm", SVC(probability=True)),
])

param_grid_pipe = {
    "svm__C": [0.01, 0.1, 1, 10, 100],
    "svm__gamma": [0.01, 0.1, 1, 10, 100],
}

grid_pipe = GridSearchCV(pipe_leak_free, param_grid=param_grid_pipe, cv=5, scoring="roc_auc")
grid_pipe.fit(X_train, y_train)

y_scores_pipe = grid_pipe.predict_proba(X_test)[:, 1]
auc_pipe = roc_auc_score(y_test, y_scores_pipe)

print(f"AUC sin fuga (con Pipeline): {auc_pipe:.3f}")
print("Mejores parámetros (sin leakage):", grid_pipe.best_params_)
print(f"\nDiferencia AUC (fuga - sin fuga): {auc_leak - auc_pipe:.3f}")


/home/yisusparker/.local/lib/python3.14/site-packages/sklearn/svm/_base.py:239: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(
/home/yisusparker/.local/lib/python3.14/site-packages/sklearn/svm/_base.py:239: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(
/home/yisusparker/.local/lib/python3.14/site-packages/sklearn/svm/_base.py:239: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(


/home/yisusparker/.local/lib/python3.14/site-packages/sklearn/svm/_base.py:239: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(
/home/yisusparker/.local/lib/python3.14/site-packages/sklearn/svm/_base.py:239: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(
/home/yisusparker/.local/lib/python3.14/site-packages/sklearn/svm/_base.py:239: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(


/home/yisusparker/.local/lib/python3.14/site-packages/sklearn/svm/_base.py:239: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(
/home/yisusparker/.local/lib/python3.14/site-packages/sklearn/svm/_base.py:239: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(
/home/yisusparker/.local/lib/python3.14/site-packages/sklearn/svm/_base.py:239: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(


/home/yisusparker/.local/lib/python3.14/site-packages/sklearn/svm/_base.py:239: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(
/home/yisusparker/.local/lib/python3.14/site-packages/sklearn/svm/_base.py:239: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(
/home/yisusparker/.local/lib/python3.14/site-packages/sklearn/svm/_base.py:239: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(


/home/yisusparker/.local/lib/python3.14/site-packages/sklearn/svm/_base.py:239: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(
/home/yisusparker/.local/lib/python3.14/site-packages/sklearn/svm/_base.py:239: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(
/home/yisusparker/.local/lib/python3.14/site-packages/sklearn/svm/_base.py:239: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(


...

/home/yisusparker/.local/lib/python3.14/site-packages/sklearn/svm/_base.py:239: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(
/home/yisusparker/.local/lib/python3.14/site-packages/sklearn/svm/_base.py:239: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(
/home/yisusparker/.local/lib/python3.14/site-packages/sklearn/svm/_base.py:239: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(


/home/yisusparker/.local/lib/python3.14/site-packages/sklearn/svm/_base.py:239: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(


/home/yisusparker/.local/lib/python3.14/site-packages/sklearn/svm/_base.py:239: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(
/home/yisusparker/.local/lib/python3.14/site-packages/sklearn/svm/_base.py:239: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(


AUC sin fuga (con Pipeline): 1.000
Mejores parámetros (sin leakage): {'svm__C': 1, 'svm__gamma': 0.1}

Diferencia AUC (fuga - sin fuga): 0.000


/home/yisusparker/.local/lib/python3.14/site-packages/sklearn/svm/_base.py:239: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(


**Observación (resultado real, ejecutado sobre `data/heart.csv`):** ambos flujos dan
**AUC = 1.000**, con o sin `Pipeline`. Esto no es un error — es el punto clave del ejercicio:
`leaky_feature` se construyó como una copia casi exacta del target (`y + ruido`), así que **ninguna**
reorganización del orden de preprocesamiento puede corregir esa fuga, porque el problema no está en
*cuándo* se ajusta el escalador, sino en que **una columna de `X` filtra literalmente el target**.

`Pipeline` + `GridSearchCV` sí protegen contra la fuga que viene de ajustar el preprocesamiento
(`scaler`, `ColumnTransformer`) con datos de prueba — como se vio en la sección 10.4 de las notas —
pero **no** pueden detectar ni corregir una feature que en sí misma encodea el target. Ese tipo de
fuga solo se evita con auditoría del dataset (revisar de dónde viene cada columna) y eliminando la
feature manualmente, que es justamente lo que se hace en la sección 6: la comparación de modelos usa
`X` **sin** `leaky_feature`, y por eso el AUC baja a un rango realista (~0.93-0.94).


## 5. Función reutilizable de entrenamiento

Encapsulamos la lógica de entrenamiento y evaluación en una función reutilizable, tal como sugiere la
sección 10.12.6 de las notas.


In [7]:
def train_pipeline(X_train, y_train, model, param_grid, preprocessor, cv=5, scoring="roc_auc"):
    """Arma un Pipeline (preprocesamiento + modelo) y ejecuta GridSearchCV sobre param_grid."""
    pipe = Pipeline([
        ("preprocessing", preprocessor),
        ("clf", model),
    ])
    grid = GridSearchCV(pipe, param_grid=param_grid, cv=cv, scoring=scoring, n_jobs=-1)
    grid.fit(X_train, y_train)
    return grid


def evaluate_pipeline(grid, X_test, y_test):
    """Devuelve AUC y Accuracy de un GridSearchCV ya ajustado, evaluado en el set de prueba."""
    y_scores = grid.predict_proba(X_test)[:, 1]
    y_pred = grid.predict(X_test)
    auc = roc_auc_score(y_test, y_scores)
    acc = accuracy_score(y_test, y_pred)
    return auc, acc


## 6. Comparación de modelos

Dividimos los datos **sin** la `leaky_feature` (dataset real y limpio) y entrenamos los 5 modelos
pedidos en la sección 10.12.6: `SVC` (el del ejemplo base) y los 4 "modelos recomendados"
(`LogisticRegression`, `RandomForestClassifier`, `KNeighborsClassifier`, `GradientBoostingClassifier`),
cada uno dentro de su propio `Pipeline` + `GridSearchCV`.


In [8]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=RANDOM_STATE, stratify=y
)

model_configs = {
    "SVC": (
        SVC(probability=True, random_state=RANDOM_STATE),
        {"clf__C": [0.1, 1, 10, 100], "clf__gamma": [0.01, 0.1, 1, 10]},
    ),
    "LogisticRegression": (
        LogisticRegression(max_iter=1000, random_state=RANDOM_STATE),
        {"clf__C": [0.01, 0.1, 1, 10, 100]},
    ),
    "RandomForestClassifier": (
        RandomForestClassifier(random_state=RANDOM_STATE),
        {"clf__n_estimators": [100, 200, 400], "clf__max_depth": [None, 5, 10]},
    ),
    "KNeighborsClassifier": (
        KNeighborsClassifier(),
        {"clf__n_neighbors": [3, 5, 7, 9, 11], "clf__weights": ["uniform", "distance"]},
    ),
    "GradientBoostingClassifier": (
        GradientBoostingClassifier(random_state=RANDOM_STATE),
        {"clf__n_estimators": [100, 200], "clf__learning_rate": [0.01, 0.1, 0.2], "clf__max_depth": [2, 3, 4]},
    ),
}

results = []
fitted_grids = {}

for name, (model, param_grid) in model_configs.items():
    grid = train_pipeline(X_train, y_train, model, param_grid, preprocessor)
    auc, acc = evaluate_pipeline(grid, X_test, y_test)
    fitted_grids[name] = grid
    results.append({
        "modelo": name,
        "AUC": auc,
        "Accuracy": acc,
        "mejores_parametros": grid.best_params_,
    })
    print(f"{name:28s} AUC={auc:.3f}  Accuracy={acc:.3f}  params={grid.best_params_}")


/home/yisusparker/.local/lib/python3.14/site-packages/sklearn/svm/_base.py:239: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(
/home/yisusparker/.local/lib/python3.14/site-packages/sklearn/svm/_base.py:239: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(
/home/yisusparker/.local/lib/python3.14/site-packages/sklearn/svm/_base.py:239: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(
/home/yisusparker/.local/lib/python3.14/site-packages/sklearn/svm/_base.py:239: FutureWarning: The `probability` parameter was deprecated in 1

/home/yisusparker/.local/lib/python3.14/site-packages/sklearn/svm/_base.py:239: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(
/home/yisusparker/.local/lib/python3.14/site-packages/sklearn/svm/_base.py:239: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(
/home/yisusparker/.local/lib/python3.14/site-packages/sklearn/svm/_base.py:239: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(
/home/yisusparker/.local/lib/python3.14/site-packages/sklearn/svm/_base.py:239: FutureWarning: The `probability` parameter was deprecated in 1

/home/yisusparker/.local/lib/python3.14/site-packages/sklearn/svm/_base.py:239: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(
/home/yisusparker/.local/lib/python3.14/site-packages/sklearn/svm/_base.py:239: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(
/home/yisusparker/.local/lib/python3.14/site-packages/sklearn/svm/_base.py:239: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(
/home/yisusparker/.local/lib/python3.14/site-packages/sklearn/svm/_base.py:239: FutureWarning: The `probability` parameter was deprecated in 1

/home/yisusparker/.local/lib/python3.14/site-packages/sklearn/svm/_base.py:239: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(


SVC                          AUC=0.941  Accuracy=0.880  params={'clf__C': 100, 'clf__gamma': 0.01}
LogisticRegression           AUC=0.933  Accuracy=0.895  params={'clf__C': 1}


RandomForestClassifier       AUC=0.944  Accuracy=0.902  params={'clf__max_depth': 5, 'clf__n_estimators': 200}


KNeighborsClassifier         AUC=0.928  Accuracy=0.880  params={'clf__n_neighbors': 11, 'clf__weights': 'uniform'}


GradientBoostingClassifier   AUC=0.937  Accuracy=0.909  params={'clf__learning_rate': 0.1, 'clf__max_depth': 2, 'clf__n_estimators': 100}


## 7. Ranking comparativo

In [9]:
ranking = (
    pd.DataFrame(results)
    .sort_values("AUC", ascending=False)
    .reset_index(drop=True)
)
ranking.index += 1
ranking


,modelo,AUC,Accuracy,mejores_parametros
1,RandomForestClassifier,0.943621,0.902174,"{'clf__max_depth': 5, 'clf__n_estimators': 200}"
2,SVC,0.940964,0.880435,"{'clf__C': 100, 'clf__gamma': 0.01}"
3,GradientBoostingClassifier,0.937404,0.909420,"{'clf__learning_rate': 0.1, 'clf__max_depth': ..."
4,LogisticRegression,0.932887,0.894928,{'clf__C': 1}
5,KNeighborsClassifier,0.927919,0.880435,"{'clf__n_neighbors': 11, 'clf__weights': 'unif..."


**Conclusión:** el modelo con mejor AUC en este ranking es el candidato a llevar a la Etapa 2
(`2_model_pipeline_cv.ipynb`), donde se hace la validación final, la evaluación completa (matriz de
confusión, curva ROC) y la exportación del modelo para servirlo con FastAPI.
